##  CELL 1 — Setup + DB Connection

In [7]:
import os
import psycopg2
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
)

print("✅ Connected to database")

✅ Connected to database


##  Inspect Problematic Records

In [8]:
query = """
SELECT id, title, published_date, url
FROM news
WHERE
      title IS NULL
   OR TRIM(title) = ''
   OR LENGTH(TRIM(title)) < 10
   OR title !~ '[A-Za-z]'
      AND title !~ '[ء-ي]'
;
"""

df_bad = pd.read_sql(query, conn)

print("Invalid title records:", len(df_bad))
df_bad.head()

Invalid title records: 253


/tmp/ipykernel_37498/3751544935.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bad = pd.read_sql(query, conn)


,id,title,published_date,url
0,174,Από τις Φλόγες στο Μπλε Υδρογόνο: Η Νέα Τάξη Π...,2026-03-28 08:50:07,https://www.naftemporiki.gr/opinion/2091463/ap...
1,181,当国家开始为文化下注：在不确定时代重构增长模型,2026-03-05 16:00:00,https://www.ftchinese.com/story/001109124
2,188,Восточный удар: цена нефти может превысить пла...,2026-03-09 17:08:41,https://iz.ru/2056113/pavel-vikhrov-albert-kal...
3,384,Κουβέιτ: Καπνοί πάνω από την αμερικανική πρεσβ...,2026-03-02 08:11:12,https://www.newsit.gr/kosmos/kouveit-kapnoi-pa...
4,7150,اختتام \,2025-10-08 22:37:00,http://webcms.aleqt.com/الأخبار/اختتام-saudi-e...


## Delete Empty Titles

In [10]:
delete_query = """
DELETE FROM news
WHERE
      title IS NULL
   OR TRIM(title) = ''
   OR LENGTH(TRIM(title)) < 10
   OR title !~ '[A-Za-z]'
      AND title !~ '[ء-ي]'
;
"""

with conn.cursor() as cur:
    cur.execute(delete_query)
    deleted = cur.rowcount
    conn.commit()

print(f"✅ Deleted {deleted} bad titles")

✅ Deleted 253 bad titles


## Verify Cleanup

In [12]:
verify_query = """
SELECT COUNT(*) AS remaining_bad_titles
FROM news
WHERE
      title IS NULL
   OR TRIM(title) = ''
   OR LENGTH(TRIM(title)) < 10
   OR (
        title !~ '[A-Za-z]'
        AND title !~ '[ء-ي]'
      );
"""

remaining = pd.read_sql(verify_query, conn)

print("Remaining bad titles:", remaining.iloc[0, 0])

Remaining bad titles: 0


/tmp/ipykernel_37498/2194920532.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  remaining = pd.read_sql(verify_query, conn)


## Close Connection

In [13]:
conn.close()
print("✅ Connection closed")

✅ Connection closed
